# Genetic Algorithm(neuroevolution) for Unit Placement

## Импорт необходимых библиотек и подготовка среды

In [7]:
pip install torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 11.8 MB/s  0:00:07 eta 0:00:010:00:01
Note: you may need to restart the kernel to use updated packages.


In [8]:
import os
import sys
import socket
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output
import time

sys.path.append(os.getcwd())

from GA.evolution import GAManager
from GA.network import decode_output

# Настройка стиля графиков
plt.style.use('ggplot')
%matplotlib inline


In [9]:
def plot_live_evolution(history):
    if not history:
        return
    
    clear_output(wait=True)
    generations = range(len(history))
    max_fitness = [h['max'] for h in history]
    avg_fitness = [h['avg'] for h in history]

    plt.figure(figsize=(12, 6))
    plt.plot(generations, max_fitness, label='Лучший результат (Max)', color='#2ca02c', marker='o', linewidth=2)
    plt.plot(generations, avg_fitness, label='Средний по популяции (Avg)', color='#1f77b4', linestyle='--')
    
    plt.title('Прогресс обучения нейроэволюции', fontsize=15)
    plt.xlabel('Поколение', fontsize=12)
    plt.ylabel('Fitness (Урон / Эффективность)', fontsize=12)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

In [13]:
import socket
import torch
import numpy as np

# Настройки эксперимента
POP_SIZE = 15
MUTATION_RATE = 0.05
HOST = '127.0.0.1'
PORT = 5005

ga = GAManager(pop_size=POP_SIZE, mutation_power=MUTATION_RATE)

def run_evolution_cycle():
    generation = 0
    
    try:
        while True:
            fitness_scores = []
            
            for i in range(POP_SIZE):
                with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
                    s.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
                    s.bind((HOST, PORT))
                    s.listen(1)
                    
                    conn, addr = s.accept()
                    with conn:
                        # 1. Прием входных данных (Бюджет|Поле)
                        raw_data = conn.recv(4096).decode().strip()
                        if not raw_data:
                            continue
                            
                        try:
                            parts = raw_data.split('|')
                            budget = float(parts[0])
                            state_vector = np.fromstring(parts[1], sep=',')
                            
                            # 2. Инференс нейросети
                            current_brain = ga.population[i]
                            with torch.no_grad():
                                state_tensor = torch.FloatTensor(state_vector)
                                probabilities = current_brain.forward(state_tensor).numpy()
                            
                            # 3. Отправка решения в Unity
                            layout_response = decode_output(probabilities, budget)
                            conn.sendall(layout_response.encode())
                            
                            # 4. Прием результата (чистый float)
                            fitness_raw = conn.recv(1024).decode().strip()
                            fitness_scores.append(float(fitness_raw))
                            
                        except (ValueError, IndexError) as e:
                            print(f"Data error at individual {i}: {e}")
                            fitness_scores.append(0.0)
            print(f"Собрано результатов: {len(fitness_scores)} из {POP_SIZE}")
            # Процесс эволюции после сбора всей популяции
            if len(fitness_scores) == POP_SIZE:
                ga.evolve(fitness_scores)
                
                # --- ГАРАНТИРОВАННЫЙ ВЫВОД В JUPYTER/VSCODE ---
                plt.close('all') # Закрываем старые невидимые окна
                clear_output(wait=True)
                
                # Создаем новую фигуру принудительно
                fig, ax = plt.subplots(figsize=(10, 5))
                
                # Вызываем отрисовку (убедись, что внутри этой функции нет plt.show())
                plot_live_evolution(ga.fitness_history)
                
                plt.show() # Явный приказ "ПОКАЖИ"
                # ---------------------------------------------
                
                print(f"Gen {generation} | Max: {max(fitness_scores):.2f} | Avg: {np.mean(fitness_scores):.2f}")
                generation += 1
            else:
                print("Error: Population size mismatch. Generation aborted.")
                break

    except KeyboardInterrupt:
        print("Training interrupted by user.")
    except Exception as e:
        print(f"System error: {e}")

# Запуск
run_evolution_cycle()

Training interrupted by user.
